# v28 — Context-DPO against recitation: the dev screen on Kaggle

The thesis's first **weight update**. The Solver (Qwen2.5-Math-7B-Instruct, 4-bit) gets a LoRA adapter trained with DPO + NLL to prefer a right solution of the real problem over **its own solution of the familiar version** (the prototype, twist removed). Rules: the `pretest_v28.py` docstring. Evidence and literature: `V28_PLAN.md`.

**What this notebook does**
1. Installs `peft` and runs the tests (`test_v28.py --require-model`, a few minutes, including LoRA DPO on a tiny model). If any fails, it stops before the long run.
2. A **smoke run** (~10-15 min): prototypes, familiar versions, 2 training pairs and a few samples, end to end on the real models. It also downloads both models once.
3. **Two workers in parallel**, one per T4. Worker *w* trains on the rows of template fold *w* and is scored on the rows of the other fold. Each stops by itself at `MAX_HOURS`.
4. Prints the pre-registered reading: **PRIMARY / ATTRIBUTE / HARM / GO**.

**Before running**: Accelerator **GPU T4 x2**, Internet **On**, secret `HF_API_KEY`. Then **Save & Run All**. Download `pretest_v28_w0.json` and `pretest_v28_w1.json` from the Output tab.

**If the session ends before the workers finish**: open a new version, **Add Input → this notebook's previous version (Output)**, and Save & Run All again. The cell copies back the two JSON files and the `v28_adapters` folder (training checkpoints included), and every worker resumes where it stopped.

Cell 1 below is Cell 1 of `MAS_SHT_Kaggle.ipynb`, unchanged.


In [ ]:
# === Cell 1 — Kaggle Setup ====================================================
# Installs deps, clones private GitHub repo via PAT, loads Kaggle Secrets.
# Re-running is safe: git pull updates the repo, pip skips installed pkgs.
#
# REQUIRED Kaggle Secrets (Add Notebook Secret in right-side panel):
#   GITHUB_TOKEN  — GitHub PAT with "Contents: read" permission
#                   (needed only if repo is private; skip if repo is public)
#   HF_API_KEY    — HuggingFace token (only for local-model runs; not needed for Groq)
#   GROQ_API_KEY  — Groq API key (REQUIRED for the Qwen3-32B Groq run)
#
# GPU: Session options → Accelerator → GPU T4 x2  (REQUIRED — v14.4 pins the two
# 4-bit models to separate GPUs; a single T4 OOMs on this preset).

import os, sys, subprocess

DEPS = [
    'openai>=1.40.0', 'sympy', 'statsmodels',
    'bitsandbytes',  # [v10.3] 4-bit quant for 7B models
    'scipy', 'pandas', 'matplotlib', 'tqdm', 'datasets',
    'transformers>=4.44.0,<4.50', 'accelerate', 'huggingface_hub',  # upper pin: an
    # unpinned install once pulled a newer transformers than the base image's torch
    # build expects, producing "module 'torch' has no attribute 'device'" on every
    # model load (2026-08-25) -- looked like a broken environment, was a version skew.
    'python-dotenv', 'tabulate', 'requests',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS)
print("Deps installed.")

# ── [v15.4] ENVIRONMENT GATE — fail in 2 minutes, not in 11 hours ────────────
# The 2026-08-29 commit ran 11 GPU-hours and produced nothing usable: the 4-bit
# load raised ImportError on BitsAndBytesConfig, the solver silently fell back
# to a full-precision path that cannot fit two 7B models on one T4, and every
# generation came back empty -- the zero-shot baseline answered 'unknown' on
# 65/65 rows. Nothing upstream noticed until the CSV was analysed the next day.
#
# The likely mechanism is that this pip install DOWNGRADES transformers (the
# Kaggle image ships a newer one) underneath a live kernel, which sometimes
# leaves the on-disk package and the imported module objects inconsistent.
# Rather than guess the one true pin, verify the exact capability the run
# depends on, right here, before anything expensive starts.
_env_errors = []
try:
    import torch as _t
    _ngpu = _t.cuda.device_count()
    print(f'  torch {_t.__version__} | cuda {_t.version.cuda} | gpus {_ngpu}')
    for _gi in range(_ngpu):
        _cc = _t.cuda.get_device_capability(_gi)
        print(f'    cuda:{_gi} {_t.cuda.get_device_name(_gi)} (sm_{_cc[0]}{_cc[1]})')
        # [v15.4] Compute capability is the check the import-level tests CANNOT
        # make: `import bitsandbytes` and `from transformers import
        # BitsAndBytesConfig` both succeed on a P100 because neither touches
        # CUDA. The incompatibility only surfaces at model-load/generate time,
        # as garbage output -- which is exactly how the 2026-08-29 session
        # burned 11 GPU-hours: Kaggle had handed it a Tesla P100 (sm_60) that
        # the installed PyTorch does not support at all, and bitsandbytes NF4
        # needs sm_75+ regardless.
        if _cc < (7, 5):
            _env_errors.append(
                f'cuda:{_gi} is {_t.cuda.get_device_name(_gi)} (sm_{_cc[0]}{_cc[1]}); '
                f'4-bit NF4 needs sm_75+ (T4 is sm_75). Imports will still succeed '
                f'on this card -- it fails silently at generate time.')
    if _ngpu < 2:
        _env_errors.append(f'need 2 GPUs for the mixed preset, found {_ngpu} '
                           f'(Session options -> Accelerator -> GPU T4 x2)')
except Exception as _e:
    _env_errors.append(f'torch unusable: {type(_e).__name__}: {_e}')
try:
    import transformers as _tf
    print(f'  transformers {_tf.__version__}')
except Exception as _e:
    _env_errors.append(f'transformers unusable: {type(_e).__name__}: {_e}')
try:
    import bitsandbytes as _bnb
    print(f'  bitsandbytes {_bnb.__version__}')
except Exception as _e:
    _env_errors.append(f'bitsandbytes not importable: {type(_e).__name__}: {_e}')
try:
    from transformers import BitsAndBytesConfig as _BNBC
    from transformers import AutoModelForCausalLM as _AMFC
    print('  BitsAndBytesConfig + AutoModelForCausalLM OK')
except Exception as _e:
    _env_errors.append(f'4-bit path unavailable: {type(_e).__name__}: {_e}')

if _env_errors:
    print('\n' + '=' * 62)
    print('ENVIRONMENT GATE: FAIL')
    for _e in _env_errors:
        print('  - ' + str(_e))
    print('\n  Every 4-bit preset in this project needs the checks above to pass.')
    print('  FIX: Session -> Restart session, then run Cell 1 FIRST in the fresh')
    print('  kernel (pip must not modify packages that are already imported).')
    print('  If it still fails, relax the transformers pin in DEPS above and')
    print('  re-run -- the image default is often the tested combination.')
    print('=' * 62)
    raise RuntimeError('Environment gate failed — see the list above. Do NOT '
                       'start a run; it would burn GPU-hours producing garbage.')
print('Environment gate: PASS')

# ── Kaggle Secrets ────────────────────────────────────────────────────────────
def _get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

_gh_token = _get_secret('GITHUB_TOKEN')   # None → assumes repo is public
_hf_token = _get_secret('HF_API_KEY')
_groq_token = _get_secret('GROQ_API_KEY')

if _hf_token:
    os.environ['HF_API_KEY']        = _hf_token
    os.environ['HUGGINGFACE_TOKEN'] = _hf_token
    os.environ['HF_TOKEN']          = _hf_token
    print("HF token loaded.")
else:
    print("WARNING: HF_API_KEY not set — model download may fail for gated repos.")

if _groq_token:
    os.environ['GROQ_API_KEY'] = _groq_token
    print("Groq token loaded.")
else:
    print("WARNING: GROQ_API_KEY not set — the Qwen3-32B Groq run WILL FAIL. Add it in Kaggle Secrets.")

# ── Clone / update repo ───────────────────────────────────────────────────────
REPO_DIR  = '/kaggle/working/llm_thesis'
REPO_NAME = 'marios4371/llm_thesis'

if _gh_token:
    REPO_URL = f'https://{_gh_token}@github.com/{REPO_NAME}.git'
else:
    REPO_URL = f'https://github.com/{REPO_NAME}.git'

if not os.path.isdir(REPO_DIR):
    print("Cloning repo…")
    result = subprocess.run(
        ['git', 'clone', REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        # Hide token from error message
        err = result.stderr.replace(_gh_token or '', '***') if _gh_token else result.stderr
        raise RuntimeError(f"git clone failed:\n{err}")
    print("Clone OK.")
else:
    print("Repo already present — pulling latest…")
    # Set the remote URL (in case token changed)
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL],
                   capture_output=True)
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', '--ff-only'],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or "Already up to date.")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Output directories ────────────────────────────────────────────────────────
ROOT          = '/kaggle/working/MAS_SHT'
RESULTS_DIR   = f'{ROOT}/results'
CHECKPOINT_DIR= f'{ROOT}/checkpoints'
ARTIFACTS_DIR = f'{ROOT}/artifacts'
for d in [RESULTS_DIR, CHECKPOINT_DIR, ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── GPU check ─────────────────────────────────────────────────────────────────
import torch
print(f"\nGPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Inference will be very slow on CPU.")
    print("Go to: Session options → Accelerator → GPU T4 x2, then restart.")

print(f"\nRepo  : {REPO_DIR}")
print(f"Results: {RESULTS_DIR}")
print(f"Checkpts: {CHECKPOINT_DIR}")


In [ ]:
# === Cell v28 — Context-DPO, dev screen (two workers, one per T4) ==================
# Pre-registered in pretest_v28.py (its docstring); plan: V28_PLAN.md. Needs Cell 1 first.
import glob, os, shutil, subprocess, sys, time

REPO_DIR  = '/kaggle/working/llm_thesis'
BRANCH   = 'Context-DPO'
MAX_HOURS = 8.0          # per worker; the batch session is capped at 9 h
OUTS      = ('pretest_v28_w0.json', 'pretest_v28_w1.json')

def _git(*args):
    r = subprocess.run(['git', '-C', REPO_DIR, *args], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stderr}")
    return r.stdout.strip()

_git('fetch', 'origin', BRANCH)
_git('checkout', '-B', BRANCH, f'origin/{BRANCH}')
print('code at', _git('log', '-1', '--format=%h %s'))
os.chdir(REPO_DIR)

# peft is the one package v28 adds; the runs below are fresh processes
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'peft>=0.12,<0.15'])

# resume from an attached previous version
for _name in OUTS:
    for _prior in sorted(glob.glob(f'/kaggle/input/**/{_name}', recursive=True))[:1]:
        shutil.copy(_prior, _name)
        print('resuming from', _prior)
for _prior in sorted(glob.glob('/kaggle/input/**/v28_adapters', recursive=True))[:1]:
    shutil.copytree(_prior, 'v28_adapters', dirs_exist_ok=True)
    print('adapters and checkpoints from', _prior)

_t = subprocess.run([sys.executable, 'test_v28.py', '--require-model'], capture_output=True, text=True)
print(_t.stdout.strip().splitlines()[-1] if _t.stdout.strip() else _t.stderr[-2000:])
if _t.returncode != 0:
    print(_t.stdout[-5000:], _t.stderr[-3000:])
    raise RuntimeError('test_v28.py failed: do NOT start the GPU run')

# smoke: the whole GPU path on 3 rows, with its own output files
_s = subprocess.run([sys.executable, '-u', 'pretest_v28.py', '--smoke', '--worker', '0', '--device', '0',
                     '--max-hours', '0.75'], capture_output=True, text=True)
print(_s.stdout[-6000:])
if _s.returncode != 0 or 'worker 0 finished' not in _s.stdout:
    print(_s.stderr[-6000:])
    raise RuntimeError('the smoke run failed: do NOT start the long run')

import torch
_two = torch.cuda.device_count() > 1
CMDS = {0: [sys.executable, '-u', 'pretest_v28.py', '--worker', '0', '--device', '0'],
        1: [sys.executable, '-u', 'pretest_v28.py', '--worker', '1', '--device', '1' if _two else '0']}

def _tail(path, n=1):
    try:
        with open(path, encoding='utf-8', errors='replace') as fh:
            return ''.join(fh.readlines()[-n:]).rstrip()
    except OSError:
        return ''

if _two:
    _procs = {}
    for w, cmd in CMDS.items():
        _procs[w] = subprocess.Popen(cmd + ['--max-hours', str(MAX_HOURS)],
                                     stdout=open(f'v28_worker{w}.log', 'a'), stderr=subprocess.STDOUT)
    while any(p.poll() is None for p in _procs.values()):
        time.sleep(300)
        for w in _procs:
            print(f'[{time.strftime("%H:%M")}] worker {w}: {_tail(f"v28_worker{w}.log")}', flush=True)
    for w, p in _procs.items():
        print(f'\n===== worker {w} (exit {p.returncode}) =====\n' + _tail(f'v28_worker{w}.log', 40))
else:
    print('ONE GPU: the workers run one after the other, each with half the time')
    for w, cmd in CMDS.items():
        with open(f'v28_worker{w}.log', 'a') as _log:
            subprocess.run(cmd + ['--max-hours', str(MAX_HOURS / 2)], stdout=_log, stderr=subprocess.STDOUT)
        print(f'\n===== worker {w} =====\n' + _tail(f'v28_worker{w}.log', 40))

!python -u pretest_v28.py --summary-only

for _name in OUTS + ('v28_worker0.log', 'v28_worker1.log'):
    if os.path.exists(_name):
        shutil.copy(_name, f'/kaggle/working/{_name}')
print('\nDownload pretest_v28_w0.json and pretest_v28_w1.json from the Output tab.')
